# 01 – Foundation (shared work, AI allowed)

**Project:** Do vision models really detect steel defects, or do they learn shortcuts?
**Dataset:** NEU Surface Defect Database (6 classes × 300 grayscale images, 200×200 px, with bounding boxes)
**Models:** ResNet-18 (ImageNet-pretrained, fine-tuned) and CLIP ViT-B/32 (zero-shot)

This notebook produces the shared starting point for both pairs:

| Output | Content |
| --- | --- |
| `resnet18_neu.pt` | fine-tuned ResNet-18 weights (not committed to git) |
| `train_history.csv` | loss / accuracy per epoch |
| `predictions.csv` | one row per image × model × prompt set × perturbation, with all 6 class probabilities |

Outputs go to `outputs/` (local) or the Drive folder `AI-Vis-Project` (Colab).

**Not in this notebook (on purpose):** visualizations, Grad-CAM, embeddings. Those are Phase I / Phase II work done by the pairs.

### Option A – run locally (no GPU needed, ~20–40 min on CPU)
1. Put `NEU-CLS.zip` in the repo's `data/` folder (or leave it one folder above the repo).
2. In Anaconda Prompt, from the repo folder: see *Setup* in `README.md`, then `jupyter lab` and open this notebook.
3. Run all cells.

### Option B – run on Google Colab (free GPU, ~10–15 min)
1. Upload `NEU-CLS.zip` and `data/split.csv` into Google Drive → *My Drive* → `AI-Vis-Project`.
2. Upload this notebook to Colab, then **Runtime → Change runtime type → T4 GPU**.
3. **Runtime → Run all** and allow Drive access.

In [ ]:
%pip install -q timm open_clip_torch

In [ ]:
import os, glob, random, zipfile, time
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageEnhance, ImageFilter
import torchvision.transforms as T
import timm, open_clip

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU (fine locally, ~20-40 min; on Colab switch to T4 GPU)")

CLASSES = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]
C2I = {c: i for i, c in enumerate(CLASSES)}

In [ ]:
# Works both in Google Colab (data in Google Drive) and locally (Jupyter in this repo folder)
import sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/AI-Vis-Project")   # change if your Drive folder has another name
    ZIP, SPLIT, OUT_DIR, DATA_DIR = BASE / "NEU-CLS.zip", BASE / "split.csv", BASE, Path("/content/neu")
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    SPLIT, OUT_DIR, DATA_DIR = REPO / "data" / "split.csv", REPO / "outputs", REPO / "data" / "raw"
    candidates = [REPO / "data" / "NEU-CLS.zip", REPO.parent / "NEU-CLS.zip"]
    ZIP = next((p for p in candidates if p.exists()), candidates[0])

NUM_WORKERS = 2 if IN_COLAB else 0   # 0 is required for notebooks on Windows
OUT_DIR.mkdir(parents=True, exist_ok=True)
assert ZIP.exists(), f"NEU-CLS.zip not found at {ZIP}"
assert SPLIT.exists(), f"split.csv not found at {SPLIT}"

if not any(DATA_DIR.glob("**/*.jpg")):
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(DATA_DIR)

paths = {p.name: str(p) for p in DATA_DIR.glob("**/images/*.jpg")}
split = pd.read_csv(SPLIT)
split["path"] = split["image"].map(paths)
assert split["path"].notna().all(), "some images in split.csv were not found in the zip"
print("Mode:", "Colab" if IN_COLAB else "local", "| outputs ->", OUT_DIR)
print(len(split), "images");  print(pd.crosstab(split.true_class, split.split))

## ResNet-18 fine-tuning

Design choice to document in the report: **no photometric augmentation** (no brightness / contrast / blur jitter). Only flips are used, so that the Phase I perturbation tests measure the model's *untrained* robustness instead of something we taught it on purpose. Flips are safe because the defect classes have no orientation.

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = T.Compose([T.Resize(224), T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf  = T.Compose([T.Resize(224), T.ToTensor(), T.Normalize(MEAN, STD)])

def load(p):
    return Image.open(p).convert("RGB")

class NEU(Dataset):
    def __init__(self, df, tf): self.df, self.tf = df.reset_index(drop=True), tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return self.tf(load(r.path)), C2I[r.true_class]

g = torch.Generator().manual_seed(SEED)
dl_train = DataLoader(NEU(split[split.split == "train"], train_tf), batch_size=32, shuffle=True, num_workers=NUM_WORKERS, generator=g)
dl_val   = DataLoader(NEU(split[split.split == "val"], eval_tf), batch_size=64, num_workers=NUM_WORKERS)

model = timm.create_model("resnet18", pretrained=True, num_classes=len(CLASSES)).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
EPOCHS = 10
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
loss_fn = nn.CrossEntropyLoss()

def evaluate(dl):
    model.eval(); correct = n = 0
    with torch.no_grad():
        for x, y in dl:
            pred = model(x.to(DEVICE)).argmax(1).cpu()
            correct += (pred == y).sum().item(); n += len(y)
    return correct / n

history, best = [], 0.0
for ep in range(1, EPOCHS + 1):
    model.train(); t0, tot = time.time(), 0.0
    for x, y in dl_train:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); loss = loss_fn(model(x), y); loss.backward(); opt.step()
        tot += loss.item() * len(y)
    sched.step()
    tr_loss, va_acc = tot / len(dl_train.dataset), evaluate(dl_val)
    history.append(dict(epoch=ep, train_loss=tr_loss, val_acc=va_acc))
    if va_acc >= best:
        best = va_acc; torch.save(model.state_dict(), f"{OUT_DIR}/resnet18_neu.pt")
    print(f"epoch {ep:2d}  train_loss {tr_loss:.4f}  val_acc {va_acc:.3f}  ({time.time()-t0:.0f}s)")

pd.DataFrame(history).to_csv(f"{OUT_DIR}/train_history.csv", index=False)
model.load_state_dict(torch.load(f"{OUT_DIR}/resnet18_neu.pt", map_location=DEVICE)); model.eval()
print("best val_acc:", best)

## Perturbations (applied to test images only, in pixel space, before normalization)

| Perturbation | Levels | Note |
| --- | --- | --- |
| brightness | 0.4 … 1.6 (factor) | PIL `ImageEnhance.Brightness` |
| contrast | 0.4 … 1.6 (factor) | PIL `ImageEnhance.Contrast` |
| blur | radius 1, 2, 3 px | Gaussian |
| noise | σ = 10, 20, 30 (0–255 scale) | Gaussian, fixed seed per image → reproducible |
| rotation | 90°, 180°, 270° | lossless, no padding |
| rotation_small | 15°, 30° | corners filled with the image's mean gray |

Levels are a starting set; the pairs can add their own perturbations or counterfactuals later.

In [ ]:
def brightness(im, f): return ImageEnhance.Brightness(im).enhance(f)
def contrast(im, f):   return ImageEnhance.Contrast(im).enhance(f)
def blur(im, r):       return im.filter(ImageFilter.GaussianBlur(r))
def rotate90(im, a):   return im.rotate(a, expand=False)
def rotate_small(im, a):
    fill = tuple(int(v) for v in np.asarray(im).reshape(-1, 3).mean(0))
    return im.rotate(a, resample=Image.BILINEAR, fillcolor=fill)
def noise(im, s, seed):
    rng = np.random.default_rng(seed)
    a = np.asarray(im).astype(np.float32) + rng.normal(0, s, np.asarray(im).shape)
    return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))

PERTURBATIONS = [("clean", 0, None)]
PERTURBATIONS += [("brightness", f, brightness) for f in (0.4, 0.6, 0.8, 1.2, 1.4, 1.6)]
PERTURBATIONS += [("contrast",   f, contrast)   for f in (0.4, 0.6, 0.8, 1.2, 1.4, 1.6)]
PERTURBATIONS += [("blur",       r, blur)       for r in (1, 2, 3)]
PERTURBATIONS += [("noise",      s, "noise")    for s in (10, 20, 30)]
PERTURBATIONS += [("rotation",   a, rotate90)   for a in (90, 180, 270)]
PERTURBATIONS += [("rotation_small", a, rotate_small) for a in (15, 30)]

def apply(im, name, level, fn, seed):
    if fn is None: return im
    if fn == "noise": return noise(im, level, seed)
    return fn(im, level)

test = split[split.split == "test"].reset_index(drop=True)
print(len(PERTURBATIONS), "conditions ×", len(test), "test images")

In [ ]:
rows = []
def add_rows(df_imgs, probs, model_name, prompt_set, pert, level):
    for (_, r), p in zip(df_imgs.iterrows(), probs):
        k = int(p.argmax())
        rows.append(dict(image=r.image, true_class=r.true_class, split=r["split"], model=model_name,
                         prompt_set=prompt_set, perturbation=pert, level=level,
                         pred_class=CLASSES[k], confidence=float(p[k]), correct=CLASSES[k] == r.true_class,
                         **{f"p_{c}": float(p[i]) for i, c in enumerate(CLASSES)}))

@torch.no_grad()
def resnet_probs(images):
    out = []
    for i in range(0, len(images), 64):
        x = torch.stack([eval_tf(im) for im in images[i:i+64]]).to(DEVICE)
        out.append(model(x).softmax(1).cpu().numpy())
    return np.concatenate(out)

# clean predictions on every split (useful for training-data hypotheses)
for s in ("train", "val"):
    d = split[split.split == s].reset_index(drop=True)
    add_rows(d, resnet_probs([load(p) for p in d.path]), "ResNet-18", "n/a", "clean", 0)

# all perturbations on the test split
base = [load(p) for p in test.path]
for name, level, fn in PERTURBATIONS:
    imgs = [apply(im, name, level, fn, seed=SEED + i) for i, im in enumerate(base)]
    add_rows(test, resnet_probs(imgs), "ResNet-18", "n/a", name, level)
print(len(rows), "ResNet rows")

## CLIP zero-shot

Three prompt sets plus their average (`ensemble`). Prompt wording is itself a Phase I variable, so it is stored in the `prompt_set` column.

In [ ]:
clip_model, _, clip_pre = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
clip_model = clip_model.to(DEVICE).eval()
tok = open_clip.get_tokenizer("ViT-B-32")

NAMES = {"crazing": "crazing", "inclusion": "inclusion", "patches": "patches",
         "pitted_surface": "pitted surface", "rolled-in_scale": "rolled-in scale", "scratches": "scratches"}
PROMPTS = {
    "plain":  "a photo of {}",
    "steel":  "a grayscale image of a steel surface with {}",
    "defect": "a close-up photo of a {} defect on hot-rolled steel",
}

@torch.no_grad()
def text_feats(template):
    t = clip_model.encode_text(tok([template.format(NAMES[c]) for c in CLASSES]).to(DEVICE))
    return t / t.norm(dim=-1, keepdim=True)

TEXT = {k: text_feats(v) for k, v in PROMPTS.items()}
ens = torch.stack(list(TEXT.values())).mean(0)
TEXT["ensemble"] = ens / ens.norm(dim=-1, keepdim=True)
for k, v in PROMPTS.items(): print(f"{k:9s} e.g. '{v.format(NAMES['pitted_surface'])}'")

@torch.no_grad()
def clip_img_feats(images):
    out = []
    for i in range(0, len(images), 64):
        f = clip_model.encode_image(torch.stack([clip_pre(im) for im in images[i:i+64]]).to(DEVICE))
        out.append(f / f.norm(dim=-1, keepdim=True))
    return torch.cat(out)

def clip_probs(img_f, txt_f):
    return (clip_model.logit_scale.exp() * img_f @ txt_f.T).softmax(1).cpu().numpy()

for s in ("train", "val"):
    d = split[split.split == s].reset_index(drop=True)
    f = clip_img_feats([load(p) for p in d.path])
    for k, t in TEXT.items():
        add_rows(d, clip_probs(f, t), "CLIP-ViT-B32", k, "clean", 0)

for name, level, fn in PERTURBATIONS:
    imgs = [apply(im, name, level, fn, seed=SEED + i) for i, im in enumerate(base)]
    f = clip_img_feats(imgs)
    for k, t in TEXT.items():
        add_rows(test, clip_probs(f, t), "CLIP-ViT-B32", k, name, level)
print(len(rows), "rows in total")

In [ ]:
pred = pd.DataFrame(rows)
pred.to_csv(f"{OUT_DIR}/predictions.csv", index=False)
print("saved", f"{OUT_DIR}/predictions.csv", pred.shape)

# sanity check only (not a Phase I result): clean test accuracy per model / prompt set
clean = pred[(pred.split == "test") & (pred.perturbation == "clean")]
print(clean.groupby(["model", "prompt_set"]).correct.mean().round(3))

## Next
- Commit `outputs/predictions.csv` and `outputs/train_history.csv` so both pairs start from the same file (Colab users: copy them into `outputs/` first). Model weights and images stay out of git.
- Log this notebook in the Record Log with *Phase = Foundation*, *Condition = Foundation (AI allowed)*.
- Phase I (pairs): build investigative visualizations on top of `predictions.csv`.
- Phase II (pairs): Grad-CAM / embeddings; `data/boxes.csv` gives the defect boxes for every image.